# Build a Minimal Deep Research — plan, research, synthesize with citations

OpenAI's **Deep Research** (and its siblings at Google and Perplexity) takes a
question, decomposes it into a research plan, spends minutes browsing and
reading sources, and returns a long-form report with citations. Under the
agentic hood, the architecture is a three-role pipeline:

```
question ──▶ PLANNER ──▶ sub-questions ──▶ RESEARCHER (×N) ──▶ findings ──▶ SYNTHESIZER ──▶ cited report
             (structured                    (tool loop over                  (writes the
              decomposition)                 a search corpus)                 final document)
```

The production systems differ from this notebook mainly in scale: live web
browsing, hundreds of sources, multi-round refinement. The role structure is
the same — and it's the role structure that makes the output good, because
each stage's output is **typed** and the next stage consumes it blind.

We rebuild it with `vidbyte-sdk` using:

- `output_schema` — the planner and researchers emit schema-validated Pydantic
  objects (`reply.metadata["structured"]`), so the pipeline composes without
  any JSON string surgery.
- `@tool` — researchers get `search` and `read_article` tools over Wikipedia's
  public API (keyless, so the notebook runs with just a model provider key).

## Setup

In [ ]:
%pip install -q vidbyte-sdk python-dotenv requests

import os
from dotenv import load_dotenv

load_dotenv()             # .env next to this notebook
load_dotenv("../../.env") # or at the repo root

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + provider overrides) in .env first."

## Step 1 — The research corpus tools

Real deep research browses the open web. We swap in Wikipedia's API to stay
keyless — the agent-facing contract (`search`, then `read_article`) is the
same shape as a production browsing tool pair.

In [ ]:
import re

import requests
from vidbyte import tool

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "vidbyte-cookbook-deep-research/0.1"}


@tool
def search(query: str) -> list[dict]:
    """Search the corpus for articles matching the query. Returns up to 5
    results with title and snippet. Use read_article to read one in full."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "list": "search", "srsearch": query,
        "srlimit": 5, "format": "json",
    })
    resp.raise_for_status()
    return [
        {"title": hit["title"], "snippet": re.sub(r"<[^>]+>", "", hit["snippet"])}
        for hit in resp.json()["query"]["search"]
    ]


@tool
def read_article(title: str) -> str:
    """Read the plain-text extract of one article by its exact title
    (truncated to ~6000 chars)."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "prop": "extracts", "explaintext": 1,
        "titles": title, "format": "json", "redirects": 1,
    })
    resp.raise_for_status()
    pages = resp.json()["query"]["pages"]
    extract = next(iter(pages.values())).get("extract", "")
    return extract[:6000] if extract else f"No article found titled '{title}'."

## Step 2 — The planner

Deep research lives or dies on decomposition: a vague question becomes 3–5
researchable sub-questions that jointly cover it. The planner's output is a
typed `ResearchPlan` — note that nothing downstream ever parses prose.

In [ ]:
from pydantic import BaseModel, Field
from vidbyte import BaseAgent


class ResearchPlan(BaseModel):
    objective: str = Field(description="One sentence stating what the report must establish.")
    sub_questions: list[str] = Field(description="3-5 independent, researchable sub-questions that jointly cover the objective.")


def structured(reply, schema):
    """Pull the schema-validated payload off a reply."""
    payload = reply.metadata.get("structured")
    if payload is None:
        raise RuntimeError(f"Output failed {schema.__name__} validation: {reply.content[:200]}")
    return payload if isinstance(payload, schema) else schema.model_validate(payload)


planner = BaseAgent(
    name="planner",
    system_prompt=(
        "You are the planning stage of a deep-research system. Decompose the "
        "user's question into 3-5 sub-questions that are independently "
        "researchable in an encyclopedia and jointly sufficient to answer it. "
        "Prefer questions about mechanisms, evidence, and counterpoints over "
        "definitions."
    ),
    provider=PROVIDER,
    model_name=MODEL,
    output_schema=ResearchPlan,
)

QUESTION = "Did the Concorde fail for economic reasons, technical reasons, or both?"

plan = structured(planner.run(QUESTION), ResearchPlan)
print("Objective:", plan.objective)
for i, q in enumerate(plan.sub_questions, 1):
    print(f"  {i}. {q}")

## Step 3 — The researchers

One researcher agent per sub-question, each running its own tool loop:
search, pick promising articles, read them, distill **source-attributed
findings**. The per-finding `source_title` is what makes the final report
citable — provenance is attached at the moment of extraction, not
reconstructed afterwards.

This stage is embarrassingly parallel; we run it as a simple loop for
readability (the SDK's `ActorRuntime` is the parallel upgrade path).

In [ ]:
class Finding(BaseModel):
    claim: str = Field(description="One specific, factual finding relevant to the sub-question.")
    source_title: str = Field(description="The exact title of the article this finding came from.")


class ResearchNotes(BaseModel):
    sub_question: str
    findings: list[Finding] = Field(description="3-6 source-attributed findings.")
    gaps: str = Field(default="", description="What the corpus could not answer, if anything.")


def research(sub_question: str) -> ResearchNotes:
    researcher = BaseAgent(
        name="researcher",
        system_prompt=(
            "You are one researcher inside a deep-research system, assigned a "
            "single sub-question. Search the corpus, read the most promising "
            "articles in full, and distill findings. Every finding must come "
            "from an article you actually read, with its exact title as the "
            "source. Record honestly what you could not establish."
        ),
        provider=PROVIDER,
        model_name=MODEL,
        tools=[search, read_article],
        max_tool_rounds=8,
        output_schema=ResearchNotes,
    )
    return structured(researcher.run(f"Research this sub-question: {sub_question}"), ResearchNotes)


notes = []
for q in plan.sub_questions:
    print("researching:", q)
    result = research(q)
    notes.append(result)
    print(f"  -> {len(result.findings)} findings\n")

## Step 4 — The synthesizer

The synthesizer never saw the corpus — it sees only the typed findings. That
constraint is deliberate and mirrors the production systems: the writer can't
introduce uncited claims, because everything it knows arrived with a source
attached.

In [ ]:
import json

from IPython.display import Markdown, display

synthesizer = BaseAgent(
    name="synthesizer",
    system_prompt=(
        "You are the synthesis stage of a deep-research system. Write a "
        "well-structured markdown report that answers the objective using "
        "ONLY the findings provided. Cite the source title in parentheses "
        "after every factual claim. Where findings conflict or gaps were "
        "reported, say so explicitly. End with a short 'Sources' list. "
        "Do not introduce facts that are not in the findings."
    ),
    provider=PROVIDER,
    model_name=MODEL,
)

payload = {
    "objective": plan.objective,
    "research_notes": [n.model_dump() for n in notes],
}
report = synthesizer.run("Write the report.\n\n" + json.dumps(payload, indent=2))
display(Markdown(report.content))

## What the production systems add

- **Live browsing** — the open web with rendering, paywalls, and PDFs instead of one clean API.
- **Iterative depth** — researchers spawn follow-up questions when findings open new threads; production runs do many rounds.
- **Source quality models** — ranking and cross-checking sources rather than trusting the first hit.
- **Parallelism** — run the researcher stage concurrently (`ActorRuntime`, or `asyncio` over `arun`).

Things to try: swap the Wikipedia tools for a real search API; let the planner
see the researchers' `gaps` fields and emit a second-round plan; make the
synthesizer's report a typed schema too and render it into HTML.